# 06 — Why the world looks classical

Every notebook so far has shown you superposition doing something a classical picture
cannot: a qubit in two states at once, a pair correlated beyond any slip of paper, a
photon taking both arms of an interferometer. All of it real, all of it measured.

So here is the obvious question, and it deserves a straight answer rather than a shrug:

> If superposition is how the world actually works, why have you never seen a coffee cup
> in two places at once?

The usual non-answer is that quantum effects are "too small to matter at everyday
scales". That cannot be right — a coffee cup is made of atoms, and every one of them
obeys the rules in notebook 01. Nothing in those rules mentions size.

The real answer is one of the most satisfying things in physics, and it needs no new
postulate, no collapse, no observer, and no boundary between a quantum world and a
classical one. **The classical world is what the quantum world looks like from the
inside**, to someone who cannot keep track of everything. That is what this notebook
builds, in code, from the gates you already have.

**What you will learn**

- What the **partial trace** is, and why it is forced rather than chosen.
- What a **mixed state** is, and how it differs from a superposition — and how it does
  *not*.
- How **decoherence** destroys interference without disturbing anything, by leaking
  information rather than by kicking the qubit.
- **Einselection**: why the states that survive are chosen by the interaction, not by
  anything intrinsic to them.
- The **quantum eraser**: decoherence run backwards, exactly.

Everything here uses one new idea — `qc.environment()` — and no new simulator machinery
at all. That is the point of the phase.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, viz
from qsim.decoherence import (
    amplitude_damping_coupling,
    dephasing_coupling,
    depolarizing_coupling,
    pointer_coupling,
)
from qsim.gates import H, Ry

np.set_printoptions(precision=3, suppress=True)

## 1. The one thing we are not allowed to do

A quantum computer's state is a single vector for *all* its qubits at once. So is the
universe's. If you want to describe a coffee cup, you are describing a subsystem — a
tiny corner of a vastly larger state that includes the air around it, the light bouncing
off it, and you.

Nobody tracks all of that. Not because it is hard, but because there is no version of
"an observer inside the universe" who could. So every description of anything is
already a description of a *part*.

That turns out to be the whole story. The rest of this notebook is about what happens
to a part.

## 2. What "not looking" means: the partial trace

We need to be precise about one operation, because everything else follows from it.

You have a joint state of two qubits, $A$ and $B$. You are going to do experiments on
$A$ only — you will never touch $B$. What is the shortest description of $A$ that still
predicts every one of those experiments correctly?

Start by grouping the terms of the joint state according to what $B$ is doing:

$$|\psi\rangle = \sum_j |\phi_j\rangle_A \otimes |j\rangle_B$$

where $\{|j\rangle_B\}$ is any basis for $B$, and $|\phi_j\rangle_A$ is whatever
$A$-vector comes along with it. These are not normalized — their lengths carry the
weights.

Now take **any** measurement $M$ on $A$. Doing nothing to $B$ means the global
observable is $M \otimes I$, and its average value is

$$\langle\psi|(M\otimes I)|\psi\rangle
 = \sum_j \langle\phi_j|M|\phi_j\rangle
 = \mathrm{Tr}\!\left(M \underbrace{\sum_j |\phi_j\rangle\langle\phi_j|}_{\rho_A}\right).$$

Look at what happened. Every prediction about $A$ — every measurement, in every basis,
now or later — depends on the state only through that one object $\rho_A$, the sum over
the index you refused to look at. That sum **is** the partial trace.

It is not an approximation, and it is not a modelling choice. It is what is left of
$|\psi\rangle$ once you ask which parts of it can still affect you. And it is unique:
two joint states with different $\rho_A$ must differ on *some* measurement of $A$, so
nothing smaller would be enough.

In `qsim` this is one line, in `inspector.py`:

```python
m = self._matricize(self._axes(subset))   # kept qubits index rows, ignored ones columns
return m @ m.conj().T                     # the matrix product sums over the ignored index
```

The matrix product's contracted index is exactly the $\sum_j$ above.

### Why the answer cannot be a state vector

For an unentangled qubit you could just read a vector off. Once $A$ is entangled with
$B$, there is no vector for $A$ to find — that is what entanglement *means*, and
notebook 02 proved it by trying to factor a Bell state and failing.

So the trace hands back a **density matrix** instead: a $2\times 2$ object rather than a
2-vector. The extra room is not waste. It is what lets one language describe both "a
genuine superposition" and "a classical coin flip" — as *different* matrices with the
same diagonal.

## 3. A qubit meets a bystander

Here is the whole mechanism, in four lines.

We put a qubit into $|+\rangle$ — a real superposition, coherence at its maximum — and
then let a *single other qubit* interact with it, gently. The interaction is
`dephasing_coupling`: conditioned on our qubit being $|1\rangle$, it rotates the other
qubit a little. Nothing is measured. Nothing is discarded. No random number is drawn.

`qc.environment(1)` allocates that other qubit and marks it. **The marking traces
nothing out** — it is a note to the Inspector saying "when I ask about *the system*, I
mean everything except this."

In [ ]:
qc = Circuit(name="bystander")
q = qc.alloc("q")
env = qc.environment(1, name="E")

H(q)                                          # q is |+>: coherence 0.5, fully quantum
dephasing_coupling(q, env[0], theta=np.pi / 3)  # one qubit takes a partial look

print("the whole state:      ", qc.inspect.ket())
print("its entropy (0 = pure):", qc.inspect.entanglement_entropy(list(qc.qubits)))

Read that output carefully, because it is the crux of the notebook.

The global state is **pure**. Entropy exactly zero. It is a perfectly definite state of
two qubits, written out in three terms, and we know it completely — nothing is unknown,
nothing is random, nothing has been lost.

Now ask the same circuit what the *system* looks like.

In [ ]:
rho = qc.inspect.system_density_matrix()   # traces out everything marked as environment

print("rho of q alone:")
print(rho)
print()
print(f"coherence |rho_01| = {qc.inspect.coherence(q):.4f}   (0.5 for a fresh |+>)")
print(f"Bloch vector       = {np.round(qc.inspect.bloch_vector(q), 4)}")
print(f"system entropy     = {qc.inspect.system_entropy():.4f} bits")

The coherence has dropped from $0.5$ to $0.433$, which is exactly
$\tfrac12\cos(\theta/2) = \tfrac12\cos(\pi/6)$. The Bloch vector has shrunk from length
1 to length $0.866$ — it is now *inside* the sphere, no longer on it. And the system has
picked up $0.35$ bits of entropy.

Nothing was done to `q`. Look at the circuit: the only gate touching `q` was the `H` at
the start. The rotation was applied to the *other* qubit.

So where did the coherence go? Into correlation. The environment qubit now leans one way
if `q` is $|0\rangle$ and another way if `q` is $|1\rangle$ — it holds a partial record
of which branch we are in. And a branch that has been recorded, even partially, even by
a single qubit nobody will ever read, is that much less able to interfere with the
others.

That is decoherence. In full.

## 4. So what *is* the difference from a pure state?

Worth pinning down, because "mixed" is a word that sounds vaguer than it is.

Take the coupling all the way to $\theta = \pi$ — a perfect record — and put the
resulting matrix next to a plain $|+\rangle$.

In [ ]:
pure = Circuit()
p = pure.alloc("p")
H(p)

mixed = Circuit()
m = mixed.alloc("m")
e = mixed.environment(1)
H(m)
dephasing_coupling(m, e[0], theta=np.pi)   # a perfect which-path record

print("|+> — a pure superposition:")
print(pure.inspect.reduced_density_matrix([p]))
print("\nthe same qubit after full decoherence:")
print(mixed.inspect.system_density_matrix())

**The diagonals are identical.** Both say: measure in the computational basis and you
get 0 half the time and 1 half the time. On that question the two qubits are
indistinguishable, and no number of repetitions will separate them.

**The off-diagonals are not.** The pure state has $\rho_{01} = 0.5$; the decohered one
has $\rho_{01} = 0$. That entry is the coherence, and it is the part that says the two
branches are still *the same state* rather than two alternatives.

The difference shows up the moment you ask a different question — one whose answer
depends on the relative phase. Measure $X$ instead of $Z$:

In [ ]:
print(f"<X> for the pure |+>       : {pure.inspect.expectation('X'):.4f}")
print(f"<X> after full decoherence : {mixed.inspect.expectation('XI'):.4f}")

$|+\rangle$ is an eigenstate of $X$: the answer is $+1$, every single time, with
certainty. The decohered qubit gives $0$ — a coin flip. Same $Z$ statistics, completely
different $X$ statistics. That is what the off-diagonal was worth.

### Two ways to get the same matrix

Now the subtle part, and it matters for the rest of the notebook.

That right-hand matrix, $\tfrac12 I$, can arise two entirely different ways:

1. Someone flipped a coin and prepared $|0\rangle$ or $|1\rangle$ without telling you.
   There is a fact; you just don't know it. (A **proper** mixture.)
2. The qubit is entangled with something you're not tracking — our case exactly. There
   is *no* fact about this qubit alone, and the global state is known perfectly.
   (An **improper** mixture.)

These are different situations, and yet **no experiment on this qubit alone can tell
them apart** — because, by section 2, every such experiment sees only $\rho$. That is
not a limitation of the description; it is the description being exactly as
discriminating as reality allows.

Here is the payoff, and hold onto it until section 9: in case 2, nothing was actually
destroyed. The information is sitting in the correlations, intact. We will go and get it
back.

## 5. Turning the knob

$\theta$ controls how much the environment learns. At $\theta = 0$ it learns nothing —
its rotation is the same either way. At $\theta = \pi$ the two branches drive it to
$|0_E\rangle$ and $|1_E\rangle$, orthogonal states, a perfect record.

In between, a partial record. Let's watch coherence follow it.

In [ ]:
thetas = np.linspace(0, np.pi, 60)
coherences = []
for theta in thetas:
    c = Circuit()
    cq = c.alloc()
    cenv = c.environment(1)
    H(cq)
    dephasing_coupling(cq, cenv[0], theta=theta)
    coherences.append(c.inspect.coherence(cq))

fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(thetas, coherences, lw=2, color="crimson", label="measured coherence")
ax.plot(thetas, 0.5 * np.cos(thetas / 2), "k--", lw=1, label=r"$\frac{1}{2}\cos(\theta/2)$")
ax.set_xlabel(r"$\theta$ — how much the environment learns")
ax.set_ylabel(r"coherence $|\rho_{01}|$")
ax.set_xticks([0, np.pi / 2, np.pi], ["0", "π/2", "π"])
ax.legend()
ax.set_title("superposition decaying as a record is written")
fig.tight_layout()

A clean $\tfrac12\cos(\theta/2)$. Not a fitted curve — the analytic answer, laid over
the measurement.

The same thing as a widget, with three views at once: the Bloch vector retracting toward
the origin, the interference visibility sliding down its curve, and the density matrix
with its off-diagonal fading while its diagonal sits perfectly still.

*Run the next cell yourself and drag the slider* — it is tagged `skip-execution`, so the
headless notebook runner steps over it. (Live widgets and a kernel with no browser
attached to it deadlock each other intermittently, which is a property of `ipywidgets`
rather than of anything here.)

In [ ]:
viz.interact_dephasing()

## 6. The two-slit experiment, in three gates

Feynman called the double slit "the only mystery" in quantum mechanics. Here it is,
complete, in three lines of qsim.

The dictionary:

| double slit | this circuit |
|---|---|
| the barrier that opens two paths | the first `H` |
| the two paths | $\lvert 0\rangle$ and $\lvert 1\rangle$ |
| a detector watching the slits | the environment qubit |
| the screen where paths recombine | the second `H` |
| bright and dark fringes | $P(0)$ and $P(1)$ |

With no detector, the two paths recombine and interfere: one outcome is certain. With a
perfect detector, the fringes vanish. And in between — this is the part the popular
account never mentions — you get *partial* fringes, degrading smoothly.

In [ ]:
print(" theta    P(0)     P(1)    visibility")
for theta in [0, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi]:
    c = Circuit()
    cq = c.alloc()
    cenv = c.environment(1)
    H(cq)                                        # open two paths
    dephasing_coupling(cq, cenv[0], theta=theta)  # let a detector watch, by degrees
    H(cq)                                        # recombine them
    populations = np.real(np.diag(c.inspect.system_density_matrix()))
    print(f"  {theta:.3f}  {populations[0]:.4f}  {populations[1]:.4f}   "
          f"{populations[0] - populations[1]:+.4f}")

Top row: the photon lands in $|0\rangle$ **every time**. Full fringes.

Bottom row: 50/50. The fringes are gone — and notice what did *not* happen to make them
go. Nobody looked at the environment qubit. No measurement was performed. No random
number was drawn; you can check that the whole run is deterministic. Merely making the
which-path information **available somewhere** was enough.

That is the sharpest statement of the mystery, and it is also its resolution.
Interference between two branches requires that there be no fact anywhere in the
universe about which one happened. Not "no observed fact" — no fact at all.

## 7. Information, not disturbance

There is a folk explanation of the double slit that says the detector *disturbs* the
photon — bumps it, kicks it, spoils its aim. It is wrong, and this simulator can show
you that it is wrong in one cell.

If the environment were kicking the qubit, the probability of finding 1 would change.
Watch it not change.

In [ ]:
c = Circuit()
cq = c.alloc("q")
cenv = c.environment(1)
Ry(cq, theta=0.7)      # a lopsided superposition, so the two populations differ

before = c.inspect.reduced_density_matrix([cq])
print(f"before: populations {np.real(np.diag(before))}   coherence {abs(before[0, 1]):.4f}")

dephasing_coupling(cq, cenv[0], theta=np.pi / 2)

after = c.inspect.system_density_matrix()
print(f"after : populations {np.real(np.diag(after))}   coherence {abs(after[0, 1]):.4f}")

The populations are **identical**, to the last digit printed and in fact to $10^{-12}$
(that is acceptance test TD4). The coherence dropped by $\cos(\pi/4)$.

So the environment did not push the qubit around. It did not add noise, in the ordinary
sense of the word. It **learned something**, and the superposition died of the learning.

Decoherence is leaked information. Once you see it that way, the rate at which a real
system decoheres stops being mysterious: it is just how fast the surroundings pick up a
record of it. A coffee cup scatters something like $10^{20}$ air molecules and photons
per second, each one carrying away a little of where the cup is. Its position is
recorded, continuously and redundantly, by the entire room. There is no chance
whatsoever of two cup-positions interfering — not because the cup is big, but because
the room already knows.

## 8. Einselection: who chooses the classical states?

Here is a question that should bother you. We keep saying $|0\rangle$ and $|1\rangle$
survive while $|+\rangle$ and $|-\rangle$ get destroyed. Why those? What makes
$|0\rangle$ the "classical" one?

Nothing does. Watch.

`pointer_coupling` is the same dephasing, conjugated into a basis of your choosing. Run
it at full strength on both $|0\rangle$ and $|+\rangle$, coupling through $z$ and
through $x$, and read off which states came through untouched — entropy 0 means "still
pure, nothing learned about it", entropy 1 bit means "completely decohered".

In [ ]:
print("            coupling through z    coupling through x")
for label, prepare in [("|0>", None), ("|+>", H)]:
    row = []
    for basis in ("z", "x"):
        c = Circuit()
        cq = c.alloc()
        cenv = c.environment(1)
        if prepare is not None:
            prepare(cq)
        pointer_coupling(cq, cenv[0], theta=np.pi, basis=basis)
        row.append(c.inspect.system_entropy())
    print(f"  {label}       {row[0]:.3f} bits            {row[1]:.3f} bits")

Exactly reversed. Couple through $z$ and $|0\rangle$ is the robust one; couple through
$x$ and $|+\rangle$ is. Same qubit, same states, same coupling strength — only the
interaction changed.

The surviving states are called **pointer states**, and the selection process is
**einselection** (environment-induced superselection). The rule is:

> The states that survive are the ones the environment cannot tell apart from
> themselves — the ones that leave *no* record when they interact.

Nothing intrinsic to $|0\rangle$ makes it classical. Its robustness is a fact about the
coupling, not about the state.

And that finally answers the coffee cup. We see definite **positions** — not definite
momenta, not definite superpositions-of-positions — because the interactions that
dominate everyday life couple through position. Light scatters off an object at a place.
Air molecules hit it at a place. Position is what the environment keeps a record of, so
position is what einselection leaves standing. A universe whose dominant interactions
coupled through something else would have a different classical world, made of different
definite things.

The classical world is not a separate realm. It is the shadow cast by whatever the
environment happens to be watching.

## 9. The quantum eraser

Everything so far might read as an elaborate way of saying "superpositions get
destroyed". They do not. Nothing here destroys anything — every operation was a
reversible unitary, and reversible means it can be run backwards.

The environment is still sitting in the circuit. Nobody threw it away. So: put the
interaction into reverse and watch the coherence come back.

In [ ]:
c = Circuit(name="eraser")
cq = c.alloc("q")
cenv = c.environment(1)
H(cq)
dephasing_coupling(cq, cenv[0], theta=np.pi)     # perfect record: coherence gone

print(f"after coupling:  coherence {c.inspect.coherence(cq):.12f}"
      f"   entropy {c.inspect.system_entropy():.12f} bits")

with c.adjoint():                                 # run that same block backwards
    dephasing_coupling(cq, cenv[0], theta=np.pi)

print(f"after erasure :  coherence {c.inspect.coherence(cq):.12f}"
      f"   entropy {c.inspect.system_entropy():.12f} bits")

H(cq)   # and the interference is back: |0> with certainty
print(f"\nfinal P(0) = {c.inspect.probabilities()[0]:.12f}")

Coherence back to $0.5$. Entropy back to $0$. Interference restored to full visibility,
and the final `H` returns $|0\rangle$ with probability 1 — to twelve decimal places
(acceptance test TD3 demands $10^{-12}$).

Nothing was repaired, because nothing had broken. The mixedness was never *in* the
qubit. It was in our decision to stop tracking the other one. Undo the correlation and
the superposition is simply there again, exactly as it was.

This is why the phase is built with `environment()` marking qubits rather than deleting
them. A simulator that implemented noise as random gates, or by throwing away a density
matrix's off-diagonals, could not run this cell — it would have destroyed the record
rather than merely looked past it, and the physics would have been quietly wrong.

### The same thing you already met

Notebook 04 made you uncompute scratch qubits, and `DirtyAncillaError` complained when
you didn't. That was this, wearing different clothes:

- A dirty ancilla **is** an environment — a qubit holding a record of which branch you
  are in.
- Uncomputation **is** erasure — running the correlation backwards so the branches can
  interfere again.
- `DirtyAncillaError` **is** the simulator refusing to let you decohere your own
  algorithm by accident.

Decoherence and failed uncomputation are one phenomenon. In notebook 08 it stops being
an analogy and starts deciding whether Shor's algorithm returns a factor or noise.

## 10. The other channels

Dephasing is the purest case — information leaks, nothing else. Two more, briefly, since
they are what real hardware actually suffers from.

**Amplitude damping** is decay: an excited qubit drops to $|0\rangle$ and the excitation
leaves. Unlike dephasing it *does* move populations, because energy really is going
somewhere.

**Depolarizing** noise is the shapeless worst case: with probability $p$ the qubit is
hit by a random one of $X$, $Y$, $Z$. It shrinks the Bloch vector toward the origin from
every direction at once.

Both are built the same way — unitary coupling, nothing random — and both state their
Kraus operators in their docstrings, which acceptance test TD6 checks independently.

In [ ]:
for name, run in [
    ("dephasing  ", lambda q, e: dephasing_coupling(q, e[0], theta=np.pi / 2)),
    ("damping    ", lambda q, e: amplitude_damping_coupling(q, e[0], theta=np.pi / 2)),
    ("depolarizing", lambda q, e: depolarizing_coupling(q, e, p=0.4)),
]:
    c = Circuit()
    cq = c.alloc()
    cenv = c.environment(2)     # two, so the depolarizing channel has room
    H(cq)
    Ry(cq, theta=0.6)           # somewhere general on the sphere
    before = np.array(c.inspect.bloch_vector(cq))
    run(cq, cenv)
    after = np.array(c.inspect.bloch_vector(cq))
    print(f"{name}  {np.round(before, 3)} -> {np.round(after, 3)}   "
          f"length {np.linalg.norm(before):.3f} -> {np.linalg.norm(after):.3f}")

Three different ways of retracting into the ball, and each has its own shape. Dephasing
pulls the vector *toward* the $z$-axis, shrinking $x$ and $y$ while leaving $z$ exactly
where it was — at $\theta = \pi$ it would land on the axis. Damping drags it toward the
north pole, because $|0\rangle$ is the ground state and that is where excitations end up.
Depolarizing shrinks it uniformly toward the centre, by a factor $1 - 4p/3$, with no
preferred direction at all.

Every one of them is a unitary on a slightly larger system, viewed from inside.

## What you now know

- **The partial trace** is forced, not chosen: it is the unique description of a
  subsystem that predicts every experiment on it. `reduced_density_matrix` is one matrix
  product, contracting the index you decided not to look at.
- A **mixed state** is what a subsystem looks like from inside. Its diagonal holds
  probabilities and its off-diagonal, the **coherence**, holds what remains of
  superposition — the part that $Z$ measurements cannot see and $X$ measurements can.
- $\tfrac12 I$ arises both from honest ignorance and from entanglement, and **no local
  experiment can tell those apart.** The density matrix is exactly as informative as
  reality permits.
- **Decoherence is leaked information, not disturbance.** Populations do not move; only
  coherences decay, at exactly the rate the environment becomes able to say which branch
  it was. TD4 pins this to $10^{-12}$.
- `qc.environment()` **traces nothing out**. The global state stays pure forever; the
  mixedness lives in the question you ask, which is why the **eraser** works and returns
  coherence exactly.
- **Einselection**: the pointer states are chosen by the coupling. The world looks like
  definite positions because position is what our environment records — not because
  position is special.
- A **dirty ancilla is an environment**, and **uncomputation is erasure**. One
  phenomenon, two notebooks.

Superposition never stopped. It spread out, into correlations with everything else, so
fast and so redundantly that no experiment you could do locally will ever see it again.
The classical world is not what is left when quantum mechanics switches off. It is what
quantum mechanics looks like when you are standing inside it.

## Next

Notebook 07 turns to the **quantum Fourier transform** — the tool that reads a periodic
pattern out of a superposition, and the engine underneath phase estimation and Shor's
algorithm. After the last two notebooks, the relevant instinct is already in place:
those algorithms work by making wrong answers interfere away, and every one of them is a
race against the decoherence you have just watched.